# Track 1 — Smart Campus Analytics — Prediction Notebook
**Team kkboys**

Loads the trained model, rebuilds the training-time features, and writes one prediction per
test row in the input order, under the column `prediction`.

Designed for the evaluation sandbox: no internet, no `train.csv`, and a test file with no ID
column and shuffled rows. Nothing here reads the training data or fits anything — every
component inside the model is already fitted.

**Model (v2):** a Ridge regression on 370 building-aware features, blended with an MLP and a
native-NaN LightGBM. Missing inputs are filled by a NaN-native sub-model imputer (one LightGBM
per gap-prone column). Rows missing `previous_usage` lean on a Ridge that was trained without it.
Every component was trained on test-matched missingness augmentation, and five seeds are averaged.
Only libraries present in the platform image are used (`numpy`, `pandas`, `scikit-learn`,
`joblib`, `lightgbm`).

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.neural_network import MLPRegressor

TEST_PATH = os.environ.get("DATATHON_INPUT_PATH", "test.csv")
OUTPUT_PATH = os.environ.get("DATATHON_OUTPUT_PATH", "predictions.csv")
print(f"input : {TEST_PATH}")
print(f"output: {OUTPUT_PATH}")

## Feature engineering

Identical to the training code (copied verbatim). The function is pure — it learns nothing
from the data — so test rows are transformed exactly as training rows were.

The per-building blocks give each building its own intercept, its own 24-hour load profile,
its own weekend offset, and its own slopes on occupancy, temperature and previous usage. The
`*_missing` flags and `n_missing` let the model know which values were observed rather than
estimated, which matters because the test set carries far more gaps than training did.

In [ ]:
RANDOM_STATE = 42

BUILDINGS = ["ADM_A", "BUS_A", "BUS_B", "ENG_A", "ENG_B", "LEC_A",
             "LIB_A", "RES_A", "RES_B", "SCI_A", "SCI_B", "SPT_A"]
DOW_MAP = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
           "Friday": 4, "Saturday": 5, "Sunday": 6}
NUMERIC = ["temperature", "humidity", "occupancy", "previous_usage"]
HELPER = ["hour", "month", "dow", "weekend", "building_code"]
DENSE = (["hour_sin", "hour_cos", "dow", "weekend", "month_sin", "month_cos"] + NUMERIC
         + [c + "_missing" for c in NUMERIC] + ["n_missing"] + [f"b_{b}" for b in BUILDINGS])


def make_features(df):
    """Pure: learns nothing from data, so train and test transform identically."""
    df = df.copy()
    X = pd.DataFrame(index=df.index)

    X["hour"] = df["hour"]
    X["month"] = df["month"]
    X["dow"] = df["day_of_week"].map(DOW_MAP)
    X["weekend"] = (X["dow"] >= 5).astype(int)
    X["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    X["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    X["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    X["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    for c in NUMERIC:
        X[c] = df[c]
        X[c + "_missing"] = df[c].isna().astype(int)
    X["n_missing"] = df[NUMERIC].isna().sum(axis=1)

    X["building_code"] = df["building_id"].map({b: i for i, b in enumerate(BUILDINGS)})

    extra = {}
    for b in BUILDINGS:
        is_b = (df["building_id"] == b).astype(float)
        extra[f"b_{b}"] = is_b
        extra[f"b_{b}_weekend"] = is_b * X["weekend"]
        for src, nm in [("occupancy", "occ"), ("temperature", "temp"), ("previous_usage", "prev")]:
            extra[f"b_{b}_{nm}"] = is_b * df[src]
        for h in range(24):
            extra[f"b_{b}_h{h}"] = is_b * (df["hour"] == h).astype(float)

    # temperature x time-of-day: cooling load depends on heat and hour jointly
    t = df["temperature"]
    extra["temp_x_hour_sin"] = t * X["hour_sin"]
    extra["temp_x_hour_cos"] = t * X["hour_cos"]
    extra["temp_sq"] = t ** 2
    extra["temp_x_occ"] = t * df["occupancy"]

    return pd.concat([X, pd.DataFrame(extra, index=df.index)], axis=1)


def without_prev(feature_names):
    """Feature list with previous_usage and every term derived from it removed."""
    return [c for c in feature_names if c != "previous_usage" and not c.endswith("_prev")]


def compact_view(Xf):
    """Raw drivers (gaps left as NaN) + calendar + building, for the native-NaN LightGBM."""
    C = Xf[["building_code", "hour", "dow", "month", "weekend"]].copy()
    for c in NUMERIC:
        C[c] = Xf[c].to_numpy()
    return C

## The model classes

These must be defined before `joblib.load`, because the saved model contains fitted instances
of them and the pickle refers to them by name. They are copied verbatim from the training code;
only their `transform` / `predict` paths run here.

* **`SubModelImputer`** fills each gap with a per-column LightGBM prediction conditioned on
  building, hour and the other drivers. A co-missing driver is passed to LightGBM as NaN rather
  than replaced by a median: in a row missing both occupancy and previous usage, a median would
  impute each from a fake value of the other.
* **`SeedMember`** holds the four components fitted on one augmented copy of the training data.
* **`BlendedEnergyModel`** averages the members and blends the components with fixed weights.
  Rows missing `previous_usage` get their own weight set.

In [ ]:
class SubModelImputer(BaseEstimator, TransformerMixin):
    """Predict each missing value with a dedicated LightGBM trained on the rows where that column
    was observed, then regenerate the derived interaction terms.

    * `extra_frame`: unlabelled rows (the test FEATURES) the per-column models may also learn from.
      Only features are used -- never the target.
    * Co-missing predictors are passed to LightGBM as NaN (native missing-value routing) rather than
      median-filled. A median is wrong in exactly the hard case: a row missing both occupancy and
      previous_usage would impute each from a fake median of the other.
    * `aug_pred`: each per-column model also trains on a copy of its data with that fraction of the
      *predictors* knocked out, so the NaN routing is learned from enough examples.
    """

    def __init__(self, extra_frame=None, aug_pred=0.15, seed=RANDOM_STATE):
        self.extra_frame = extra_frame
        self.aug_pred = aug_pred
        self.seed = seed

    def fit(self, X, y=None):
        src = X if self.extra_frame is None else pd.concat([X, self.extra_frame], ignore_index=True)
        rng = np.random.default_rng(self.seed)
        self.models_ = {}
        for c in NUMERIC:
            others = [o for o in NUMERIC if o != c]
            feats = HELPER + others
            observed = src[c].notna()
            Xt, yt = src.loc[observed, feats].copy(), src.loc[observed, c]
            if self.aug_pred > 0:
                Xd = Xt.copy()
                for o in others:
                    Xd.loc[rng.random(len(Xd)) < self.aug_pred, o] = np.nan
                Xt, yt = pd.concat([Xt, Xd]), pd.concat([yt, yt])
            m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31,
                                  random_state=RANDOM_STATE, verbose=-1, n_jobs=-1)
            self.models_[c] = (m.fit(Xt, yt), feats)
        return self

    def transform(self, X):
        X = X.copy()
        raw = X[NUMERIC].copy()
        for c in NUMERIC:
            gaps = raw[c].isna()
            if not gaps.any():
                continue
            m, feats = self.models_[c]
            others = [o for o in NUMERIC if o != c]
            tmp = X.loc[gaps, feats].copy()
            tmp[others] = raw.loc[gaps, others]          # co-missing predictors stay NaN
            X.loc[gaps, c] = m.predict(tmp)

        # derived terms were built from the raw (gappy) columns -- rebuild them
        for b in BUILDINGS:
            is_b = X[f"b_{b}"]
            for src, nm in [("occupancy", "occ"), ("temperature", "temp"), ("previous_usage", "prev")]:
                X[f"b_{b}_{nm}"] = is_b * X[src]
        X["temp_sq"] = X["temperature"] ** 2
        X["temp_x_occ"] = X["temperature"] * X["occupancy"]
        X["temp_x_hour_sin"] = X["temperature"] * X["hour_sin"]
        X["temp_x_hour_cos"] = X["temperature"] * X["hour_cos"]
        return X.fillna(0.0)


class SeedMember(BaseEstimator):
    """The four components fitted on one augmented copy of the training data, sharing one imputer."""

    def __init__(self, extra_frame=None, seed=0):
        self.extra_frame = extra_frame
        self.seed = seed

    def fit(self, Xf, y, Xf_lgbm):
        feats = list(Xf.columns)
        self.imputer_ = SubModelImputer(extra_frame=self.extra_frame, seed=RANDOM_STATE + self.seed).fit(Xf)
        F = self.imputer_.transform(Xf)
        ridge = lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3.5, 30)))
        self.cols_ = {"ridge": feats, "ridge_no_prev": without_prev(feats), "mlp": DENSE}
        self.ridge_ = ridge().fit(F[self.cols_["ridge"]], y)
        self.ridge_no_prev_ = ridge().fit(F[self.cols_["ridge_no_prev"]], y)
        self.mlps_ = [make_pipeline(StandardScaler(), MLPRegressor(
                          hidden_layer_sizes=(128, 64), alpha=1e-3, max_iter=600, early_stopping=True,
                          random_state=100 * self.seed + s)).fit(F[DENSE], y) for s in range(3)]
        self.lgbm_ = lgb.LGBMRegressor(
            n_estimators=2500, learning_rate=0.01, num_leaves=15, subsample=0.8, subsample_freq=1,
            colsample_bytree=0.8, random_state=RANDOM_STATE + self.seed, verbose=-1, n_jobs=-1,
        ).fit(compact_view(Xf_lgbm), y, categorical_feature=["building_code"])
        return self

    def predict_components(self, Xf):
        F = self.imputer_.transform(Xf)
        return {
            "ridge": self.ridge_.predict(F[self.cols_["ridge"]]),
            "ridge_no_prev": self.ridge_no_prev_.predict(F[self.cols_["ridge_no_prev"]]),
            "mlp": np.mean([m.predict(F[DENSE]) for m in self.mlps_], axis=0),
            "lgbm": self.lgbm_.predict(compact_view(Xf)),
        }


class BlendedEnergyModel(BaseEstimator):
    """Seed-averaged components, blended with fixed weights; rows missing previous_usage use their own
    weight set, which leans on the Ridge that never saw previous_usage."""

    def __init__(self, members, weights, feature_names):
        self.members = members
        self.weights = weights
        self.feature_names = feature_names

    def predict(self, Xf):
        Xf = Xf[self.feature_names]
        parts = [m.predict_components(Xf) for m in self.members]
        comp = {k: np.mean([p[k] for p in parts], axis=0) for k in parts[0]}
        prev_missing = Xf["previous_usage_missing"].to_numpy() == 1
        pred = np.empty(len(Xf))
        for group, mask in [("prev_observed", ~prev_missing), ("prev_missing", prev_missing)]:
            pred[mask] = sum(w * comp[k][mask] for k, w in self.weights[group].items())
        return pred

## Load, predict, write

`model.pkl` holds the fitted `BlendedEnergyModel` and the exact feature list used in training.

In [ ]:
model, features = joblib.load("model.pkl")
print(f"loaded {type(model).__name__}: {len(model.members)} seed members, {len(features)} features")

test_df = pd.read_csv(TEST_PATH)
print(f"test rows: {len(test_df):,}  columns: {list(test_df.columns)}")

X_test = make_features(test_df)

missing = [c for c in features if c not in X_test.columns]
if missing:
    raise RuntimeError(f"feature mismatch — missing {len(missing)}: {missing[:5]}")
X_test = X_test[features]

preds = model.predict(X_test)

In [ ]:
# --- validate before writing -------------------------------------------------
assert len(preds) == len(test_df), f"row count mismatch: {len(preds)} vs {len(test_df)}"
assert np.isfinite(preds).all(), "non-finite predictions"
if (preds < 0).any():
    n_neg = int((preds < 0).sum())
    print(f"clipping {n_neg} negative predictions to 0 (energy cannot be negative)")
    preds = np.clip(preds, 0, None)

pd.DataFrame({"prediction": preds}).to_csv(OUTPUT_PATH, index=False)
print(f"wrote {len(preds)} predictions to {OUTPUT_PATH}")
print(pd.Series(preds).describe().round(3).to_string())